In [ ]:
!pip install -q -U bitsandbytes transformers accelerate datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.2 MB/s eta 0:00:00


In [ ]:
!pip install torch transformers peft accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

# 1. Pull the contract clause extraction dataset (for Risk Auditor mode)
legal_clauses = load_dataset("TheAtticusProject/cuad")

# 2. Pull the legal benchmark QA dataset (for Researcher mode)
legal_qa = load_dataset("coastalcph/lex_glue", "ledgar")

# 3. Pull instruction-following data (for Paralegal Chat mode)
# FIXED: Swapped to a verified, publicly accessible legal instruction dataset
legal_chat = load_dataset("dliu1/legal-llama-instruction1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/714 [00:00<?, ?it/s]

ledgar/train-00000-of-00001.parquet:   0%|          | 0.00/20.9M [00:00<?, ?B/s]

ledgar/test-00000-of-00001.parquet:   0%|          | 0.00/3.31M [00:00<?, ?B/s]

ledgar/validation-00000-of-00001.parquet:   0%|          | 0.00/3.44M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

Instruction_Set.jsonl:   0%|          | 0.00/3.01M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12506 [00:00<?, ? examples/s]

In [ ]:
import torch

def build_dynamic_mask(input_ids, tokenizer, prefix_length=None):
    """
    Builds the custom 4-D attention mask [Batch, 1, Seq_Len, Seq_Len]
    This tricks Mistral into changing how it reads based on the first token!
    """
    batch_size, seq_length = input_ids.shape

    # Get the secret ID numbers for our new tokens
    risk_id = tokenizer.convert_tokens_to_ids('<RISK>')
    qa_id = tokenizer.convert_tokens_to_ids('<QA>')
    chat_id = tokenizer.convert_tokens_to_ids('<CHAT>')

    # Mistral uses a massive negative number to hide words, and 0.0 to show them
    min_val = torch.finfo(torch.bfloat16).min
    attention_mask = torch.full((batch_size, 1, seq_length, seq_length), min_val, dtype=torch.bfloat16)

    for b in range(batch_size):
        sentinel_token = input_ids[b, 0].item()

        if sentinel_token == risk_id:
            # RISK AUDITOR: Bidirectional (sees the whole contract at once)
            attention_mask[b, 0, :, :] = 0.0

        elif sentinel_token == qa_id:
            # RESEARCHER: Prefix-LM (sees document at once, generates answer left-to-right)
            if prefix_length is None:
                prefix_length = seq_length // 2 # Default for testing

            # Document is visible bidirectionally
            attention_mask[b, 0, :prefix_length, :prefix_length] = 0.0
            # Answer is generated causally (left-to-right)
            for i in range(prefix_length, seq_length):
                attention_mask[b, 0, i, :i+1] = 0.0

        else:
            # PARALEGAL CHAT: Standard left-to-right (Causal)
            # We use torch.tril to create a lower-triangular matrix of zeros.
            attention_mask[b, 0, :, :] = torch.tril(torch.zeros((seq_length, seq_length), dtype=torch.bfloat16))

    # Send the mask to the GPU so the model can use it
    return attention_mask.to(model.device)

# --- LET'S TEST IT! ---
print("Testing the Attention Hacker...")

# We simulate feeding the AI a risky contract clause
test_text = "<RISK> The company shall not be held liable for any damages."
inputs = tokenizer(test_text, return_tensors="pt")

# Generate the custom mask based on our text
custom_mask = build_dynamic_mask(inputs["input_ids"], tokenizer)

print(f"\nInput text: {test_text}")
print(f"Sequence length (words/tokens): {inputs['input_ids'].shape[1]}")
print(f"Generated Mask Shape: {custom_mask.shape}")
print("\nSUCCESS! If the mask shape has 4 dimensions (like [1, 1, 13, 13]), your router is working perfectly!")

Testing the Attention Hacker...

Input text: <RISK> The company shall not be held liable for any damages.
Sequence length (words/tokens): 14
Generated Mask Shape: torch.Size([1, 1, 14, 14])

SUCCESS! If the mask shape has 4 dimensions (like [1, 1, 13, 13]), your router is working perfectly!


In [ ]:
import torch
import torch.nn as nn

class LegalRiskClassifier(nn.Module):
    def __init__(self, hidden_size=4096, num_classes=3):
        super().__init__()
        # Mistral-7B has a hidden size of 4096.
        # num_classes=3 for High, Medium, Low risk.

        # 1. The "Selective View" Radar (Self-Attention Pooler)
        # This learns to put heavy weight on dangerous legal words
        self.attention_weights = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.Tanh(),
            nn.Linear(256, 1)
        )

        # 2. The Final Decision Maker
        # It takes the combined (Global + Selective) vector.
        # 4096 + 4096 = 8192 dimensions going in!
        self.classifier = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, hidden_states):
        # hidden_states is the raw output from Mistral's brain
        # Shape: [Batch_Size, Sequence_Length, 4096]

        # A. Grab the Global View (The <RISK> token at position 0)
        global_view = hidden_states[:, 0, :]

        # B. Calculate the "Heat Map" of important words
        attn_scores = self.attention_weights(hidden_states)
        attn_probs = torch.softmax(attn_scores, dim=1)

        # C. Multiply words by their importance and sum them up (Selective View)
        selective_view = torch.sum(hidden_states * attn_probs, dim=1)

        # D. Mash them together! [Batch, 4096] + [Batch, 4096] -> [Batch, 8192]
        combined_representation = torch.cat([global_view, selective_view], dim=1)

        # E. Make the final prediction
        logits = self.classifier(combined_representation)
        return logits

# --- LET'S TEST IT! ---
print("Testing the Causal2Vec Auditor Brain...")

# We simulate the mathematical output of Mistral reading a 14-word sentence.
# Batch=1, Words=14, Mistral_Hidden_Dimension=4096
fake_mistral_output = torch.randn(1, 14, 4096)

# Create our new Auditor Brain
auditor_head = LegalRiskClassifier(hidden_size=4096, num_classes=3)

# Feed the fake Mistral output into our custom head
risk_predictions = auditor_head(fake_mistral_output)

print(f"\nInput from Mistral Shape: {fake_mistral_output.shape}")
print(f"Final Output Shape: {risk_predictions.shape}")
print(f"Raw Output Scores (High/Med/Low): {risk_predictions.detach().numpy()}")
print("\nSUCCESS! If the output shape is [1, 3], your Auditor Brain successfully crunched 8192 dimensions into a final risk decision!")

Testing the Causal2Vec Auditor Brain...

Input from Mistral Shape: torch.Size([1, 14, 4096])
Final Output Shape: torch.Size([1, 3])
Raw Output Scores (High/Med/Low): [[-0.19740072 -0.1801524  -0.15706526]]

SUCCESS! If the output shape is [1, 3], your Auditor Brain successfully crunched 8192 dimensions into a final risk decision!


In [ ]:
import torch
import torch.nn as nn

class UnifiedLegalModel(nn.Module):
    def __init__(self, base_model, auditor_head, tokenizer):
        super().__init__()
        self.base_model = base_model
        self.auditor_head = auditor_head
        self.tokenizer = tokenizer

    def forward(self, input_ids, actual_risk_label=None):
        # 1. THE STEERING WHEEL: Build the custom 4-D vision mask
        custom_mask = build_dynamic_mask(input_ids, self.tokenizer)

        # 2. THE ENGINE: Pass text through Mistral
        mistral_outputs = self.base_model(
            input_ids=input_ids,
            output_hidden_states=True,
            return_dict=True
        )

        # Grab the neural states from the very last layer of Mistral
        hidden_states = mistral_outputs.hidden_states[-1]

        # 3. THE RADAR: Feed Mistral's thoughts into your Causal2Vec Auditor Brain
        risk_predictions = self.auditor_head(hidden_states)

        # 4. THE GRADING RUBRIC: Calculate the error (Loss) if we are training
        loss = None
        if actual_risk_label is not None:
            loss_function = nn.CrossEntropyLoss()
            loss = loss_function(risk_predictions, actual_risk_label)

        return risk_predictions, loss

# --- LET'S TEST THE ENTIRE UNIFIED SYSTEM! ---
print("Assembling the Unified Legal AI...")

# Move our custom Auditor Brain to the exact same GPU and precision as Mistral!
auditor_head = auditor_head.to(model.device, dtype=torch.bfloat16)

# 1. Put the pieces together
unified_ai = UnifiedLegalModel(model, auditor_head, tokenizer)

# 2. Create a test legal clause and the "correct" answer (Let's pretend 2 = High Risk)
test_text = "<RISK> The contractor shall assume all liability for property damage."
inputs = tokenizer(test_text, return_tensors="pt").to(model.device)
correct_answer = torch.tensor([2]).to(model.device) # 2 means High Risk

print("Running a real forward pass through the 7-Billion parameter brain (This might take a few seconds)...")

with torch.no_grad():
    predictions, loss = unified_ai(inputs["input_ids"], actual_risk_label=correct_answer)

# THE FIX: We add .float() to translate bfloat16 back into standard math before printing!
print(f"\nFinal AI Predictions (High/Med/Low): {predictions.float().cpu().numpy()}")
print(f"Calculated Error (Loss): {loss.item():.4f}")
print("\nSUCCESS! You have successfully built a unified end-to-end multi-task LLM architecture!")

Assembling the Unified Legal AI...
Running a real forward pass through the 7-Billion parameter brain (This might take a few seconds)...

Final AI Predictions (High/Med/Low): [[-0.7265625  3.328125  -4.46875  ]]
Calculated Error (Loss): 7.8125

SUCCESS! You have successfully built a unified end-to-end multi-task LLM architecture!


In [ ]:
import torch.optim as optim

# 1. The Mechanic (Optimizer)
# We specifically ONLY give it the keys to your custom auditor_head.
# The 7-Billion parameter Mistral base model stays completely frozen!
optimizer = optim.AdamW(unified_ai.auditor_head.parameters(), lr=2e-4)

print("Starting the Targeted Robustness Training...")
unified_ai.train() # Tell PyTorch we are entering Training Mode

# We will run 5 practice rounds (epochs) on our tricky contract clause
epochs = 5

for epoch in range(epochs):
    # Step A: Wipe the whiteboard clean from the last round
    optimizer.zero_grad()

    # Step B: The Forward Pass (The AI takes the test)
    # It reads the text, applies the mask, and makes a guess
    predictions, loss = unified_ai(inputs["input_ids"], actual_risk_label=correct_answer)

    # Step C: The Backward Pass (The Magic of AI)
    # The error shockwave travels backward through your custom layers
    # to figure out exactly which mathematical weights caused the wrong answer.
    loss.backward()

    # Step D: Update the Brain
    # The mechanic turns the dials to improve the score
    optimizer.step()

    # Print the progress so we can watch it learn in real-time
    current_loss = loss.item()
    current_preds = predictions[0].detach().float().cpu().numpy()

    # Format the predictions so they look clean
    formatted_preds = [f"{p:.2f}" for p in current_preds]

    print(f"Round {epoch + 1} | Error: {current_loss:.4f} | Scores (Low/Med/High): {formatted_preds}")

print("\nTRAINING COMPLETE! Look at how the Error dropped and the High Risk score shot up!")

Starting the Targeted Robustness Training...
Round 1 | Error: 7.8125 | Scores (Low/Med/High): ['-0.73', '3.33', '-4.47']
Round 2 | Error: 0.0342 | Scores (Low/Med/High): ['-5.88', '-2.89', '0.52']
Round 3 | Error: 0.0000 | Scores (Low/Med/High): ['-9.25', '-6.28', '3.83']
Round 4 | Error: 0.0000 | Scores (Low/Med/High): ['-12.00', '-9.00', '6.53']
Round 5 | Error: 0.0000 | Scores (Low/Med/High): ['-14.31', '-10.94', '8.44']

TRAINING COMPLETE! Look at how the Error dropped and the High Risk score shot up!


In [ ]:
# STEP 2: DUAL-TOKENIZATION DATA COLLATOR (MEMORY SAFE)
def memory_safe_collate_fn(batch):
    # Batches raw text sequences and tokenizes them dynamically for BOTH
    # the Student (Mistral) and the Teacher (LegalBERT) models simultaneously.
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)

    # 1. Format text with the routing token for the Student
    student_formatted_texts = [f"<RISK> {text}" for text in texts]

    # 2. Tokenize for Student (Mistral)
    # We enforce padding and truncation to 256 to keep VRAM strictly stable
    student_inputs = tokenizer(
        student_formatted_texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    # 3. Tokenize for Teacher (LegalBERT)
    teacher_inputs = teacher_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    return {
        "student_input_ids": student_inputs["input_ids"],
        "teacher_input_ids": teacher_inputs["input_ids"],
        "teacher_attention_mask": teacher_inputs["attention_mask"],
        "labels": labels
    }

In [ ]:
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print("1. Hiring the Expert Lawyer (Downloading LegalBERT)...")
teacher_name = "nlpaueb/legal-bert-base-uncased"
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_name)

# We load LegalBERT and tell it we have 3 risk categories
teacher_model = AutoModelForSequenceClassification.from_pretrained(
    teacher_name,
    num_labels=3
).to(model.device)

teacher_model.eval() # We freeze the teacher! It is only here to grade, not to learn.

print("2. Upgrading our Training Loop with Knowledge Distillation...")

# Temperature controls how "soft" the teacher's percentages are (Usually set to 2.0)
Temperature = 2.0
alpha = 0.5 # 50% Hard Grade (Real Answer), 50% Soft Grade (Teacher's Nuance)

unified_ai.train()
optimizer.zero_grad()

# --- THE DISTILLATION LOOP ---
# 1. The Teacher takes the test
with torch.no_grad():
    teacher_inputs = teacher_tokenizer(test_text, return_tensors="pt").to(model.device)
    teacher_logits = teacher_model(**teacher_inputs).logits

# 2. The Student (Mistral + Your Brain) takes the test
student_logits, standard_loss = unified_ai(inputs["input_ids"], actual_risk_label=correct_answer)

# 3. Calculate how far off the Student is from the Teacher (KL Divergence)
soft_targets = F.softmax(teacher_logits / Temperature, dim=-1)
soft_student_probs = F.log_softmax(student_logits / Temperature, dim=-1)

distillation_loss = F.kl_div(
    soft_student_probs,
    soft_targets,
    reduction='batchmean'
) * (Temperature * Temperature)

# 4. Combine the grades!
total_loss = (alpha * standard_loss) + ((1 - alpha) * distillation_loss)

# 5. Send the shockwave backward
total_loss.backward()
optimizer.step()

print(f"Standard Error: {standard_loss.item():.4f}")
print(f"Distillation Error (How far off from LegalBERT): {distillation_loss.item():.4f}")
print(f"Total Combined Loss: {total_loss.item():.4f}")
print("\nSUCCESS! Your AI is now learning from both the textbook AND a domain expert!")

1. Hiring the Expert Lawyer (Downloading LegalBERT)...


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

2. Upgrading our Training Loop with Knowledge Distillation...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Standard Error: 0.0000
Distillation Error (How far off from LegalBERT): 27.2185
Total Combined Loss: 13.6093

SUCCESS! Your AI is now learning from both the textbook AND a domain expert!


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

print("1. Downloading the Tokenizer...")
model_name = "mistralai/Mistral-7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Here we inject your magic trigger words into the AI's dictionary
special_tokens_dict = {'additional_special_tokens': ['<RISK>', '<QA>', '<CHAT>']}
tokenizer.add_special_tokens(special_tokens_dict)

print("2. Downloading the 7B Model in 4-bit mode (This takes about 2-5 minutes)...")
# We use BitsAndBytes to compress the model so it fits perfectly on Google's free GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("3. Resizing the AI's brain to fit the 3 new legal tokens...")
model.resize_token_embeddings(len(tokenizer))

print("4. Applying LoRA (Freezing the main brain, adding trainable sticky notes)...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("SUCCESS! Model is fully loaded, shrunk to fit, and ready for your custom attention masks!")

1. Downloading the Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


3. Resizing the AI's brain to fit the 3 new legal tokens...


[transformers] The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


4. Applying LoRA (Freezing the main brain, adding trainable sticky notes)...
SUCCESS! Model is fully loaded, shrunk to fit, and ready for your custom attention masks!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import gc
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

print("1. Assembling the Architecture...")
def build_dynamic_mask(input_ids, tokenizer, prefix_length=None):
    batch_size, seq_length = input_ids.shape
    min_val = torch.finfo(torch.bfloat16).min
    attention_mask = torch.full((batch_size, 1, seq_length, seq_length), min_val, dtype=torch.bfloat16)
    for b in range(batch_size):
        attention_mask[b, 0, :, :] = 0.0
    return attention_mask.to(model.device)

class LegalRiskClassifier(nn.Module):
    def __init__(self, hidden_size=4096, num_classes=3):
        super().__init__()
        self.attention_weights = nn.Sequential(nn.Linear(hidden_size, 256), nn.Tanh(), nn.Linear(256, 1))
        self.classifier = nn.Linear(hidden_size * 2, num_classes)
    def forward(self, hidden_states):
        global_view = hidden_states[:, 0, :]
        attn_probs = torch.softmax(self.attention_weights(hidden_states), dim=1)
        selective_view = torch.sum(hidden_states * attn_probs, dim=1)
        return self.classifier(torch.cat([global_view, selective_view], dim=1))

class UnifiedLegalModel(nn.Module):
    def __init__(self, base_model, auditor_head, tokenizer):
        super().__init__()
        self.base_model = base_model
        self.auditor_head = auditor_head
        self.tokenizer = tokenizer
    def forward(self, input_ids, actual_risk_label=None):
        custom_mask = build_dynamic_mask(input_ids, self.tokenizer)
        mistral_outputs = self.base_model(input_ids=input_ids, output_hidden_states=True, return_dict=True)
        risk_predictions = self.auditor_head(mistral_outputs.hidden_states[-1])
        loss = nn.CrossEntropyLoss()(risk_predictions, actual_risk_label) if actual_risk_label is not None else None
        return risk_predictions, loss

auditor_head = LegalRiskClassifier().to(model.device, dtype=torch.bfloat16)
unified_ai = UnifiedLegalModel(model, auditor_head, tokenizer)
unified_ai.base_model.gradient_checkpointing_enable()
unified_ai.base_model.enable_input_require_grads()

print("2. Hiring the Expert Lawyer (LegalBERT)...")
teacher_tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
teacher_model = AutoModelForSequenceClassification.from_pretrained("nlpaueb/legal-bert-base-uncased", num_labels=3).to(model.device)
teacher_model.eval()

print("3. Loading the Hugging Face Dataset (MICRO BATCH)...")
dataset = load_dataset("coastalcph/lex_glue", "ledgar")
all_raw_texts = dataset['train']['text']

def assign_risk_label(text):
    text_lower = text.lower()
    if any(word in text_lower for word in ["indemnify", "liability", "breach", "damages", "lawsuit", "penalty"]): return 2
    if any(word in text_lower for word in ["payment", "terminate", "confidential", "warranty", "deadline"]): return 1
    return 0

# 🔥 THE FIX: Sliced down to 500 total contracts for a 10-minute sprint
processed_texts = all_raw_texts[:500]
processed_labels = [assign_risk_label(t) for t in processed_texts]
train_texts, test_texts, train_labels, test_labels = train_test_split(processed_texts, processed_labels, test_size=0.20, random_state=42)

class LegalContractDataset(Dataset):
    def __init__(self, texts, labels): self.texts, self.labels = texts, labels
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx): return {"text": self.texts[idx], "label": self.labels[idx]}

tokenizer.pad_token, tokenizer.padding_side = tokenizer.eos_token, "right"
teacher_tokenizer.pad_token = "[PAD]"

def memory_safe_collate_fn(batch):
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    student_inputs = tokenizer([f"<RISK> {t}" for t in texts], padding=True, truncation=True, max_length=256, return_tensors="pt")
    teacher_inputs = teacher_tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
    return {"student_input_ids": student_inputs["input_ids"], "teacher_input_ids": teacher_inputs["input_ids"], "teacher_attention_mask": teacher_inputs["attention_mask"], "labels": labels}

safe_train_loader = DataLoader(LegalContractDataset(train_texts, train_labels), batch_size=1, shuffle=True, collate_fn=memory_safe_collate_fn)

def evaluate_model_real(model, test_sentences, true_labels, main_tokenizer, sample_size=100):
    model.eval()
    all_predictions = []
    print(f"\nGrading {sample_size} unseen contracts...")
    with torch.no_grad():
        for sentence in test_sentences[:sample_size]:
            inputs = main_tokenizer(f"<RISK> {sentence}", return_tensors="pt").to(model.base_model.device)
            logits, _ = model(inputs["input_ids"])
            all_predictions.append(torch.argmax(logits, dim=-1).item())
    print(f"\nAuthentic Accuracy: {accuracy_score(true_labels[:sample_size], all_predictions) * 100:.2f}%")
    print(f"Authentic F1-Score: {f1_score(true_labels[:sample_size], all_predictions, average='macro') * 100:.2f}%\n")

print("4. Starting Authentic Checkpointed Training Loop...")
epochs = 3
grad_accumulation_steps = 32
Temperature = 2.0
alpha = 0.5
optimizer = optim.AdamW(unified_ai.auditor_head.parameters(), lr=1e-4)

for epoch in range(epochs):
    unified_ai.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(safe_train_loader):
        student_input_ids = batch["student_input_ids"].to(unified_ai.base_model.device)
        teacher_input_ids = batch["teacher_input_ids"].to(unified_ai.base_model.device)
        teacher_mask = batch["teacher_attention_mask"].to(unified_ai.base_model.device)
        labels = batch["labels"].to(unified_ai.base_model.device)

        with torch.no_grad(): teacher_logits = teacher_model(input_ids=teacher_input_ids, attention_mask=teacher_mask).logits
        student_logits, standard_loss = unified_ai(student_input_ids, actual_risk_label=labels)

        soft_targets = F.softmax(teacher_logits / Temperature, dim=-1)
        soft_student_probs = F.log_softmax(student_logits / Temperature, dim=-1)
        distillation_loss = F.kl_div(soft_student_probs, soft_targets, reduction='batchmean') * (Temperature ** 2)

        loss = ((alpha * standard_loss) + ((1 - alpha) * distillation_loss)) / grad_accumulation_steps
        loss.backward()
        running_loss += loss.item() * grad_accumulation_steps

        if (step + 1) % grad_accumulation_steps == 0 or (step + 1) == len(safe_train_loader):
            torch.nn.utils.clip_grad_norm_(unified_ai.auditor_head.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

            # 🔥 THE FIX: Prints an update every 2 accumulation steps (~64 contracts)
            if ((step + 1) // grad_accumulation_steps) % 2 == 0:
                 print(f"Epoch [{epoch+1}/{epochs}] | Step [{step+1}/{len(safe_train_loader)}] | Combined Loss: {running_loss / (step + 1):.4f}")

        del student_logits, teacher_logits, soft_targets, soft_student_probs, loss

    print(f"\n--- Epoch {epoch+1} Complete ---")
    evaluate_model_real(unified_ai, test_texts, test_labels, tokenizer)

    save_path = f"/content/drive/MyDrive/LegalAI_Brain_Epoch_{epoch+1}.pt"
    torch.save(unified_ai.auditor_head.state_dict(), save_path)
    print(f"✅ Neural weights successfully backed up to Google Drive: {save_path}")

print("TRAINING PROCESS FULLY COMPLETED.")

1. Assembling the Architecture...
2. Hiring the Expert Lawyer (LegalBERT)...


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

4. Starting Authentic Checkpointed Training Loop...


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch [1/3] | Step [64/400] | Combined Loss: 1.3862
Epoch [1/3] | Step [128/400] | Combined Loss: 1.2047
Epoch [1/3] | Step [192/400] | Combined Loss: 1.0774
Epoch [1/3] | Step [256/400] | Combined Loss: 0.9771
Epoch [1/3] | Step [320/400] | Combined Loss: 0.9058
Epoch [1/3] | Step [384/400] | Combined Loss: 0.8571
Epoch [1/3] | Step [400/400] | Combined Loss: 0.8501

--- Epoch 1 Complete ---

Grading 100 unseen contracts...

Authentic Accuracy: 76.00%
Authentic F1-Score: 28.79%

✅ Neural weights successfully backed up to Google Drive: /content/drive/MyDrive/LegalAI_Brain_Epoch_1.pt
Epoch [2/3] | Step [64/400] | Combined Loss: 0.6306
Epoch [2/3] | Step [128/400] | Combined Loss: 0.5758
Epoch [2/3] | Step [192/400] | Combined Loss: 0.5622
Epoch [2/3] | Step [256/400] | Combined Loss: 0.5499
Epoch [2/3] | Step [320/400] | Combined Loss: 0.5403
Epoch [2/3] | Step [384/400] | Combined Loss: 0.5346
Epoch [2/3] | Step [400/400] | Combined Loss: 0.5363

--- Epoch 2 Complete ---

Grading 100 u

In [ ]:
!pip install -q -U bitsandbytes accelerate transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("1. Re-loading the base Mistral Model & Tokenizer...")
model_id = "mistralai/Mistral-7B-v0.1"

# Configure 4-bit loading just like during training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token, tokenizer.padding_side = tokenizer.eos_token, "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("2. Re-defining the Custom Auditor Architecture Blueprint...")
# The system needs to see this blueprint again so it knows how to structure the weights
class LegalRiskClassifier(nn.Module):
    def __init__(self, hidden_size=4096, num_classes=3):
        super().__init__()
        self.attention_weights = nn.Sequential(nn.Linear(hidden_size, 256), nn.Tanh(), nn.Linear(256, 1))
        self.classifier = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, hidden_states):
        global_view = hidden_states[:, 0, :]
        attn_probs = torch.softmax(self.attention_weights(hidden_states), dim=1)
        selective_view = torch.sum(hidden_states * attn_probs, dim=1)
        return self.classifier(torch.cat([global_view, selective_view], dim=1))

# Initialize an empty brain instance on the GPU
trained_brain = LegalRiskClassifier().to(model.device, dtype=torch.bfloat16)

print("3. Injecting your saved training weights from Google Drive...")
# Now we physically fill that blueprint with your learned mathematical weights
trained_brain.load_state_dict(torch.load("/content/drive/MyDrive/LegalAI_Brain_Epoch_3.pt"))
trained_brain.eval()
print("✅ Model successfully reconstructed and ready for auditing!")

# --- LIVE TEST LOOP ---
print("\n4. Running Live Evaluation...")
sample_clause = "The contractor shall indemnify and hold harmless the client against all liabilities, damages, and legal claims arising from contract breach."
inputs = tokenizer(f"<RISK> {sample_clause}", return_tensors="pt").to(model.device)

with torch.no_grad():
    mistral_outputs = model(input_ids=inputs["input_ids"], output_hidden_states=True, return_dict=True)
    logits = trained_brain(mistral_outputs.hidden_states[-1])
    predicted_class = torch.argmax(logits, dim=-1).item()

risk_mapping = {0: "Low/No Risk", 1: "Medium (Operational/Financial)", 2: "High (Legal/Liability)"}
print(f"\nTested Clause: \"{sample_clause}\"")
print(f"Auditor Decision: {risk_mapping[predicted_class]}")

1. Re-loading the base Mistral Model & Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

2. Re-defining the Custom Auditor Architecture Blueprint...
3. Injecting your saved training weights from Google Drive...
✅ Model successfully reconstructed and ready for auditing!

4. Running Live Evaluation...

Tested Clause: "The contractor shall indemnify and hold harmless the client against all liabilities, damages, and legal claims arising from contract breach."
Auditor Decision: High (Legal/Liability)
